# Query Data with AWS Data Wrangler

**AWS Data Wrangler** is an open-source Python library that extends the power of the Pandas library to AWS connecting DataFrames and AWS data related services (Amazon Redshift, AWS Glue, Amazon Athena, Amazon EMR, Amazon QuickSight, etc).

* https://github.com/awslabs/aws-data-wrangler
* https://aws-data-wrangler.readthedocs.io

Built on top of other open-source projects like Pandas, Apache Arrow, Boto3, s3fs, SQLAlchemy, Psycopg2 and PyMySQL, it offers abstracted functions to execute usual ETL tasks like load/unload data from Data Lakes, Data Warehouses and Databases.

_Note that AWS Data Wrangler is simply a Python library that uses existing AWS Services.  AWS Data Wrangler is not a separate AWS Service.  You install AWS Data Wrangler through `pip install` as we will see next._

# _Pre-Requisite: Make Sure You Created an Athena Table for Both TSV and Parquet in Previous Notebooks_

In [1]:
%store -r ingest_create_athena_table_tsv_passed

In [2]:
try:
    ingest_create_athena_table_tsv_passed
except NameError:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not register the TSV Data.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")

In [3]:
print(ingest_create_athena_table_tsv_passed)

True


In [4]:
if not ingest_create_athena_table_tsv_passed:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not register the TSV Data.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")
else:
    print("[OK]")

[OK]


In [5]:
%store -r ingest_create_athena_table_parquet_passed

In [6]:
try:
    ingest_create_athena_table_parquet_passed
except NameError:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not convert into Parquet data.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")

In [7]:
print(ingest_create_athena_table_parquet_passed)

True


In [8]:
if not ingest_create_athena_table_parquet_passed:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not convert into Parquet data.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")
else:
    print("[OK]")

[OK]


# Setup

In [9]:
import sagemaker
import boto3

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sm = boto3.Session().client(service_name="sagemaker", region_name=region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [10]:
import awswrangler as wr

# Query Parquet from S3 with Push-Down Filters

Read Apache Parquet file(s) from from a received S3 prefix or list of S3 objects paths.

The concept of Dataset goes beyond the simple idea of files and enable more complex features like partitioning and catalog integration (AWS Glue Catalog): 

_dataset (bool)_ – If True read a parquet dataset instead of simple file(s) loading all the related partitions as columns.

In [11]:
p_filter = lambda x: x["product_category"] == "Digital_Software"

In [14]:
path = "s3://{}/amazon-reviews-pds/parquet/".format(bucket)
df_parquet_results = wr.s3.read_parquet(
    path, columns=["star_rating", "product_category", "review_body"], partition_filter=p_filter, dataset=True
)
df_parquet_results.shape

(102084, 3)

In [15]:
df_parquet_results.head(5)

,star_rating,review_body,product_category
0,4,So far so good,Digital_Software
1,3,Needs a little more work.....,Digital_Software
2,1,Please cancel.,Digital_Software
3,5,Works as Expected!,Digital_Software
4,4,I've had Webroot for a few years. It expired a...,Digital_Software


# Query Parquet from S3 in Chunks

Batching (chunked argument) (Memory Friendly):

Will enable the function to return a Iterable of DataFrames instead of a regular DataFrame.

There are two batching strategies on Wrangler:
* If chunked=True, a new DataFrame will be returned for each file in your path/dataset.
* If chunked=INTEGER, Wrangler will iterate on the data by number of rows equal to the received INTEGER.

P.S. chunked=True if faster and uses less memory while chunked=INTEGER is more precise in number of rows for each Dataframe.

In [16]:
path = "s3://{}/amazon-reviews-pds/parquet/".format(bucket)
chunk_iter = wr.s3.read_parquet(
    path,
    columns=["star_rating", "review_body"],
    # filters=[("product_category", "=", "Digital_Software")],
    partition_filter=p_filter,
    dataset=True,
    chunked=True,
)

In [17]:
print(next(chunk_iter))

       star_rating                                        review_body  \
0                4                                     So far so good   
1                3                      Needs a little more work.....   
2                1                                     Please cancel.   
3                5                                 Works as Expected!   
4                4  I've had Webroot for a few years. It expired a...   
...            ...                                                ...   
65531            1  i downloaded this and recognized as i was work...   
65532            2  I upgraded from 2011 and wished I hadn't.  Eve...   
65533            4  i think this was one of my best buys for 2012....   
65534            5  I've been using quicken now for about 10 years...   
65535            5  Fast, easy download. Easy to use software. Man...   

       product_category  
0      Digital_Software  
1      Digital_Software  
2      Digital_Software  
3      Digital_Soft

# Query the Glue Catalog (ie. Hive Metastore)
Get an iterator of tables.

In [18]:
database_name = "dsoaws"
table_name_tsv = "amazon_reviews_tsv"
table_name_parquet = "amazon_reviews_parquet"

In [19]:
for table in wr.catalog.get_tables(database="dsoaws"):
    print(table["Name"])

amazon_reviews_parquet
amazon_reviews_tsv


# Query from Athena
Execute any SQL query on AWS Athena and return the results as a Pandas DataFrame.  


In [20]:
%%time
df = wr.athena.read_sql_query(sql="SELECT * FROM {} LIMIT 5000".format(table_name_parquet), database=database_name)

/opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


CPU times: user 1.55 s, sys: 294 ms, total: 1.85 s
Wall time: 4.88 s


In [21]:
df.head(5)

,marketplace,customer_id,review_id,product_id,product_parent,product_title,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,year,review_date,product_category
0,US,25836040,R1I0O8GQWODVFB,B009WD26EO,90187391,Amazon.com $70 Gift Card in a Greeting Card (A...,5,0,0,N,Y,Five Stars,So very pleased!,2015,2015-08-30,Gift Card
1,US,3561787,R2KY64Z4IBFUZS,B004LLIL9Q,667798887,Amazon eGift Card - Seasonal (Summer Pinwheels),5,0,0,N,Y,Five Stars,Easy and awesome,2015,2015-08-30,Gift Card
2,US,47984088,R1WTIOQ2FAXLW3,B00BWDH3VS,473048287,Amazon.com eGift Cards,5,0,0,N,Y,Perfect gift idea!,They loved it!,2015,2015-08-30,Gift Card
3,US,26330189,RAIHZTLAQ4JQ1,B00US9QTGM,298664776,Amazon Allowance,5,0,0,N,Y,Easy and awesome,Great way to share and/or reward!!,2015,2015-08-30,Gift Card
4,US,29608955,R2198ZQY69V35A,B004LLIL5U,864052097,Amazon eGift Card - Graduation,5,0,0,N,Y,Five Stars,You can never go wrong with Amazon gift card,2015,2015-08-30,Gift Card


# Query from Athena in Chunks
Retrieving in chunks can help reduce memory requirements.  

_This will take a few seconds._

In [22]:
%%time

chunk_iter = wr.athena.read_sql_query(
    sql="SELECT * FROM {} LIMIT 5000".format(table_name_parquet),
    database="{}".format(database_name),
    chunksize=64_000,  # 64 KB Chunks
)

/opt/conda/lib/python3.12/site-packages/awswrangler/athena/_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


CPU times: user 842 ms, sys: 213 ms, total: 1.05 s
Wall time: 3.84 s


In [23]:
print(next(chunk_iter))

     marketplace customer_id       review_id  product_id product_parent  \
0             US    25836040  R1I0O8GQWODVFB  B009WD26EO       90187391   
1             US     3561787  R2KY64Z4IBFUZS  B004LLIL9Q      667798887   
2             US    47984088  R1WTIOQ2FAXLW3  B00BWDH3VS      473048287   
3             US    26330189   RAIHZTLAQ4JQ1  B00US9QTGM      298664776   
4             US    29608955  R2198ZQY69V35A  B004LLIL5U      864052097   
...          ...         ...             ...         ...            ...   
4995          US     5323682   RM9KM7ILUO20Z  B0066AZGD4      136017760   
4996          US      181518   RUBS1SXHOD37H  BT00CTOYI4      384589818   
4997          US    46857809   RTLGMDSSTZCPZ  B00A48G0D4      848703272   
4998          US    11186745   RO30ZW3P7HGN6  B00A4EK4CQ      363923416   
4999          US    45832925  R2IEX0HYN21T47  B00IX1I3G6      926539283   

                                          product_title  star_rating  \
0     Amazon.com $70 Gift C

# Release Resources

In [24]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

I0000 00:00:1789347460.827811    3906 chttp2_transport.cc:1182] ipv4:169.255.255.2:34503: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {created_time:"2026-09-14T00:57:40.82780809+00:00", http2_error:2, grpc_status:14}
*** SIGTERM received at time=1789347463 on cpu 0 ***
PC: @     0x7feaa1685172  (unknown)  epoll_wait
[2026-09-14 00:57:43,302 E 3586 3586] logging.cc:474: *** SIGTERM received at time=1789347463 on cpu 0 ***
[2026-09-14 00:57:43,302 E 3586 3586] logging.cc:474: PC: @     0x7feaa1685172  (unknown)  epoll_wait


In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}